# Neural Network Regression: Reproducible Development Set

Change the hyperparameters in the configuration cell and compare development MAE. The split remains fixed across runs.

In [ ]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
# Hyperparameters: change these values and rerun from this cell onward.
DEV_SIZE = 0.20
BATCH_SIZE = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.25
MAX_EPOCHS = 100
PATIENCE = 15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'whatmakestitle'
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
texts = (train_df['title'].fillna('').astype(str) + ' [DOMAIN] ' + train_df['domain'].fillna('').astype(str)).tolist()
test_texts = (test_df['title'].fillna('').astype(str) + ' [DOMAIN] ' + test_df['domain'].fillna('').astype(str)).tolist()
targets = train_df['interest'].astype('float32').to_numpy()
print(f'Train rows: {len(train_df)} | test rows: {len(test_df)}')

## BERT embeddings

The transformer is used as a frozen feature extractor. The configurable neural-network head is trained on the fixed development split.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased').to(device).eval()

def embed_texts(texts, batch_size=16):
    embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            embeddings.append(bert(**batch).last_hidden_state[:, 0, :].cpu())
    return torch.cat(embeddings).float()

X = embed_texts(texts)
X_test = embed_texts(test_texts)
print(f'Embedding shape: {tuple(X.shape)} | device: {device}')

## Train and evaluate on the fixed development set

In [ ]:
class InterestRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(), nn.Dropout(dropout / 2),
            nn.Linear(hidden_dims[1], 1))

    def forward(self, features):
        return self.network(features)

train_idx, dev_idx = train_test_split(
    np.arange(len(train_df)), test_size=DEV_SIZE, random_state=SEED,
    stratify=train_df['interest'])
train_dataset = TensorDataset(X[train_idx], torch.from_numpy(targets[train_idx]).unsqueeze(1))
dev_dataset = TensorDataset(X[dev_idx], torch.from_numpy(targets[dev_idx]).unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
dev_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

model = InterestRegressor(X.shape[1], HIDDEN_DIMS, DROPOUT).to(device)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_weights, best_dev_mae = None, float('inf')
no_improvement = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        criterion(model(batch_x), batch_y).backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        dev_predictions = model(X[dev_idx].to(device)).squeeze(1).cpu().numpy()
    dev_mae = mean_absolute_error(targets[dev_idx], np.clip(dev_predictions, 1, 6))
    if dev_mae < best_dev_mae:
        best_dev_mae, best_weights, no_improvement = dev_mae, copy.deepcopy(model.state_dict()), 0
    else:
        no_improvement += 1
    if (epoch + 1) % 10 == 0 or no_improvement == 0:
        print(f'Epoch {epoch + 1:03d} | development MAE: {dev_mae:.4f}')
    if no_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch + 1}')
        break

model.load_state_dict(best_weights)
torch.save(model.state_dict(), DATA_DIR / 'best_interest_regressor.pt')
print(f'Best development MAE: {best_dev_mae:.4f}')

## Final fit and submission

After selecting hyperparameters, this uses the best development-selected weights to score the test rows.

In [ ]:
model.eval()
with torch.no_grad():
    test_predictions = model(X_test.to(device)).squeeze(1).cpu().numpy()
submission = pd.DataFrame({'title_id': test_df['title_id'], 'interest': np.clip(test_predictions, 1, 6)})
submission_path = DATA_DIR / 'submission_nn.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
submission.head()